# Purpose
The goal of this notebook is to demonstrate how to access, download, and review Roman SNANA output files and summary information. It is divided into three sections:

I. Explore the summary information of the most recent campaign, also accessible on https://roman-snpit-snana-strategy.lbl.gov

II. Download the data via API

III. Review the Roman SNANA simulations

In [1]:
import numpy as np
import pandas as pd
from astropy.io import fits
from get_fits import download_fits, get_hdu, get_data
from pathlib import Path
from roman_snana_api import Roman_SNANA_Summary

# I. Explore Roman SNANA Summary Information

You'll frequently see the term "campaign" being used. This is used to collectively describe many hundreds of SNANA simulations performed at once. Let's take a look at the most recent campaign:

In [2]:
client = Roman_SNANA_Summary()
client.get_campaigns()

,Campaign Name
0,2024-08-05_x108_3SNrates


Within a campaign exists several different collections of simulations that are defined by broad parameters. For example:

- How many tiers of photometric observations there are.
- What fraction of the time goes to the prism.
- What photometric bands are used.

A collection also defines a 3d grid of values for three parameters:

- The relative sky area covered by each tier.
- The cadence in days of each tier.
- The target redshift (where SNR tries to be 100) for each tier.

Let's begin by identifying the various collections available within the most recent campaign:

In [3]:
campaign = "2024-08-05_x108_3SNrates"
client.get_collections(campaign)

,Collections
0,2TIER_RATE0
1,2TIER_RATE1
2,2TIER_RATE2
3,3TIER_RATE0
4,3TIER_RATE1
5,3TIER_RATE2
6,3TIER_SPECIAL_RATE0
7,3TIER_SPECIAL_RATE1
8,3TIER_SPECIAL_RATE2


In [4]:
# select a collection and review the keys containing summary information:
collection = "2TIER_RATE0" # enter a collection identified in the cell above
client.get_summarydata_keys(campaign, collection)

['status',
 'campaign',
 'collection',
 'surveyinfo',
 'instrinfo',
 'analysisinfo',
 'tiers',
 'surveys']

The value of the key: `surveys` returns a JSON-encoded dictionary with a lot of information about each collection. The key is the name of the sim, and the value is another dictionary with a lot of information. The name of the sim should probably not be algorithmically parsed, but it's always '{collection} a{ai}_t{ti}_z{zi}' where `ai`, `ti`, and `zi` are indexes into the arrays defined in `tiers` above.

By specifying the campaign, collection, and key, we can begin to explore the details associated with these properties.

In [5]:
key = "surveyinfo" # select a key identified in the cell above
client.get_summarydata(campaign, collection, key)

{'OUTDIR': 'output_2TIER_RATE0',
 'FORCE_TEXPOSE_LIST': '60 75 100 150 200 300 400 600 1000',
 'FORCE_SIMGEN_INPUT_FILE': 'INP_SIMGEN_ROMAN_PEAKONLY.INPUT',
 'FORCE_NGEN': 100,
 'FORCE_SNRMAX': [{'snr': 2.0, 'lam0': 1000.0, 'lam1': 3000.0},
  {'snr': 8.0, 'lam0': 3000.0, 'lam1': 4000.0},
  {'snr': 10.0, 'lam0': 4000.0, 'lam1': 25000.0}],
 'NLIBID_TOT': 100,
 'TIME_SUM_OBS': 135,
 'TEXPOSE_MIN': 10,
 'RANDOM_REJECT_OBS': 0.125,
 'MJD_SEASON': [{'season_mjd0': 55000.0, 'season_mjd1': 55365.0},
  {'season_mjd0': 55365.0, 'season_mjd1': 55725.0}],
 'TIERS': ['SHALLOW   10  -10   RZYJ    [1,2,4]    [8, 8, 8 ]   [0.5]',
  'DEEP      20  +10   YJHF    [1,1,1]    [5, 8, 12]   [1.2]'],
 'TEXPOSE_PRISM': {'SHALLOW': [1000, 2000, 3000], 'DEEP': [3000, 6000, 10000]}}

In [6]:
# alternatively, this information can also be obtained using the following method
client.get_survey_info_keys(campaign, collection)

['OUTDIR',
 'FORCE_TEXPOSE_LIST',
 'FORCE_SIMGEN_INPUT_FILE',
 'FORCE_NGEN',
 'FORCE_SNRMAX',
 'NLIBID_TOT',
 'TIME_SUM_OBS',
 'TEXPOSE_MIN',
 'RANDOM_REJECT_OBS',
 'MJD_SEASON',
 'TIERS',
 'TEXPOSE_PRISM']

In [7]:
key = "OUTDIR" # swap 'OUTDIR' with any of the sub-dictionary keys listed above
client.get_survey_info(campaign, collection, key)

,OUTDIR
0,output_2TIER_RATE0


Let's see what simulations are available under this current collection:

In [8]:
client.get_sim_names(campaign, collection)

['2TIER_RATE0 a00-t00-z00',
 '2TIER_RATE0 a00-t01-z00',
 '2TIER_RATE0 a00-t02-z00',
 '2TIER_RATE0 a01-t00-z00',
 '2TIER_RATE0 a01-t01-z00',
 '2TIER_RATE0 a01-t02-z00',
 '2TIER_RATE0 a02-t00-z00',
 '2TIER_RATE0 a02-t01-z00',
 '2TIER_RATE0 a02-t02-z00']

In [9]:
# explore the data
sim = "2TIER_RATE0 a00-t00-z00"
client.get_sim_keys(campaign, collection, sim)

dict_keys(['tiers', 'gentypemap', 'detectedzhist', 'snrmaxzhist', 'snrmax2zhist', 'snrmax3zhist', 'long_survey_version', 'spechists', 'muopt'])

As shown above, the sub-dictionary for a single sim has keys:

- `tiers`: Information about the tiers for this particular sim 
- `gentypemap`: A mapping of SNANA gentypes (used elsewhere) to actual types
- `detectedzhist`: Numbers of objects found as function of tier, redshift and gentype
- `snrmaxzhist`
- `snrmax2zhist`
- `snrmax3zhist`
- `long_survey_version`: (used internally, may be ignored)
- `spechists`: A complicated dictionary with information about numbers of spectra (see below)
- `muopt`: (may be ignored)

You can see the value of each key using the cell below:

In [10]:
key = "gentypemap" # swap 'gentypemap' with any of the sub-dictionary keys listed above
client.get_sim_info(campaign, collection, sim, key)

{'10': 'Ia',
 '32': 'IIP',
 '33': 'IIL',
 '21': 'Ib',
 '26': 'Ic',
 '11': 'SNIa-91bg',
 '42': 'TDE',
 '40': 'SLSN-I',
 '60': 'AGN'}

The cells below contain additional examples that may be useful:

In [11]:
# a list with information about the tiers from this collection; ghe length of the list is the number of tiers
client.get_tiers(campaign, collection)

['SHALLOW', 'DEEP']

In [12]:
# returns a JSON-encoded dictionary with bunch of information about the simulated instrument that SNANA used
client.get_instrument_info_keys(campaign, collection)

['PARAMS',
 'FILTER_WAVELENGTH',
 'READ_NOISE',
 'READ_NOISE_H18_OBSOLETE',
 'THERMAL',
 'ZODIAC',
 'ZEROPOINT',
 'NEA',
 'NEA_WEBBPSF',
 'NEA_2021_UNKNOWN']

In [13]:
key = "PARAMS" # swap 'PARAMS' with any of the sub-dictionary keys listed above
client.get_instrument_info(campaign, collection, key)

,PARAMS
PIXSIZE,0.110
FOV,0.281
TIME_SLEW,70.000


In [14]:
# a JSON-encoded dictionary with a bunch of parameters that defined what SNANA did
client.get_analysisinfo_keys(campaign, collection)

['NCORE_MAX', 'SIM', 'LCFIT', 'BBC', 'WFIT', 'prescales']

In [15]:
key = "prescales" # swap 'prescales' with any of the sub-dictionary keys listed above
client.get_analysisinfo(campaign, collection, key)

{'IIP': 10.0,
 'IIL': 10.0,
 'Ib': 10.0,
 'Ic': 10.0,
 'AGN': 100.0,
 'SLSN-I': 0.3,
 'IIP+IIL': 10.0}

# Download the Data via API

In order to download SNANA simulations from a particular collection, you'll need to specify the names of the simulations. Let's remind ourselves how to access that information:

In [16]:
campaign = "2024-08-05_x108_3SNrates"
collection = "2TIER_RATE0" # update if necessary
client.get_sim_names(campaign, collection)

['2TIER_RATE0 a00-t00-z00',
 '2TIER_RATE0 a00-t01-z00',
 '2TIER_RATE0 a00-t02-z00',
 '2TIER_RATE0 a01-t00-z00',
 '2TIER_RATE0 a01-t01-z00',
 '2TIER_RATE0 a01-t02-z00',
 '2TIER_RATE0 a02-t00-z00',
 '2TIER_RATE0 a02-t01-z00',
 '2TIER_RATE0 a02-t02-z00']

Executing the following cell will download Roman SNANA files (FITS format) into the directory `fits_dump`. Executing the cell will create subdirectories within `fits_dump` that depend on the specified collection and index.

In [17]:
# define parameters using the information explored previously
index = "a00-t00-z00" # select target index
fits_type = ["PHOT", "HEAD"] # can select PHOT, HEAD, and/or SPEC
model = ["NONIa", "Ia"] # can select NONIa and/or Ia
collection = "2TIER_RATE0"

# download the entire dataset of a particular collection & index
for item in fits_type:
    for model_type in model:
        download_fits(collection, index, item, model_type)

# Review the Roman SNANA simulations

In [18]:
fits_dump_dir = "fits_dump"
fits_dir = Path.cwd() / f"{fits_dump_dir}/{collection}/{index}"

# open an individual FITS file and review its content
fits_filepath = f"{fits_dir}/ROMAN_NONIaMODEL1-0001_PHOT.FITS"
phot_hdul = fits.open(fits_filepath)
phot_hdul.info()

Filename: /Users/pitt-googlebroker/Desktop/work/roman/aas/fits_dump/2TIER_RATE0/a00-t00-z00/ROMAN_NONIaMODEL1-0001_PHOT.FITS
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      45   (0,)      
  1  Photometry    1 BinTableHDU     63   17191R x 18C   [1D, 2A, 1I, 12A, 1J, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E, 1E]   


In [19]:
# primary HDU for PHOT file
get_hdu(phot_hdul, name="PRIMARY")

,Keyword,Value
0,SIMPLE,True
1,BITPIX,-32
2,NAXIS,1
3,NAXIS1,0
4,EXTEND,True
5,COMMENT,FITS (Flexible Image Transport System) forma...
6,COMMENT,"and Astrophysics', volume 376, page 359; bib..."
7,CODE_IVERSION,25
8,SNANA_PATH,/project2/rkessler/PRODUCTS/SNANA/SNANA
9,SNANA_VERSION,v11_05o-42-g39479d4


In [20]:
# first extension HDU for PHOT file
get_hdu(phot_hdul, name="Photometry")

,Keyword,Value
0,XTENSION,BINTABLE
1,BITPIX,8
2,NAXIS,2
3,NAXIS1,80
4,NAXIS2,17191
...,...,...
58,TUNIT17,
59,TTYPE18,SIM_MAGOBS
60,TFORM18,1E
61,TUNIT18,
